In [1]:
import requests
import pandas as pd
import os
from time import sleep

def extrair_ifdata_robusto(pasta_destino):
    os.makedirs(pasta_destino, exist_ok=True)
    
    tipo_inst = 1
    relatorios = {
        "1": "Resumo", 
        "2": "Ativo", 
        "3": "Passivo", 
        "4": "Resultado"
    }
    
    anos = range(2014, 2025)
    trimestres = ["03", "06", "09", "12"]
    
    print(f"Iniciando Extração de Alta Performance: {os.path.abspath(pasta_destino)}")
    print("-" * 60)

    session = requests.Session()
    session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"})

    for ano in anos:
        for tri in trimestres:
            data_ref = int(f"{ano}{tri}")
            
            for cod_rel, nome_rel in relatorios.items():
                nome_arquivo = f"IFDATA_{data_ref}_{nome_rel}.csv"
                caminho_completo = os.path.join(pasta_destino, nome_arquivo)

                if os.path.exists(caminho_completo):
                    print(f"[PULANDO] {nome_arquivo} já existe.")
                    continue
                
                url = (
                    f"https://olinda.bcb.gov.br/olinda/servico/IFDATA/versao/v1/odata/"
                    f"IfDataValores(AnoMes=@AnoMes,TipoInstituicao=@TipoInstituicao,Relatorio=@Relatorio)"
                    f"?@AnoMes={data_ref}&@TipoInstituicao={tipo_inst}&@Relatorio='{cod_rel}'&$format=json"
                )
                
                sucesso = False
                
                for tentativa in range(1, 4):
                    try:
                        resposta = session.get(url, timeout=120)
                        
                        if resposta.status_code == 200:
                            dados = resposta.json().get('value', [])
                            if dados:
                                df = pd.DataFrame(dados)
                                df.to_csv(caminho_completo, index=False, sep=';', encoding='utf-8-sig')
                                print(f"[OK] Sucesso: {data_ref} - {nome_rel} ({len(df)} linhas)")
                            else:
                                print(f"[-] Vazio: {data_ref} - {nome_rel}")
                                
                            sucesso = True
                            break 
                            
                        else:
                            print(f"[Erro {resposta.status_code}] Tentativa {tentativa}/3 para {nome_rel}_{data_ref}")
                            sleep(3) 
                            
                    except requests.exceptions.ReadTimeout:
                        print(f"[Timeout] BCB lento na tentativa {tentativa}/3 ({nome_rel}_{data_ref}). Retentando...")
                        sleep(5)
                    except requests.exceptions.RequestException as e:
                        print(f"[Falha de Rede] Tentativa {tentativa}/3: {e}")
                        sleep(5)
                
                if not sucesso:
                    print(f"[DESISTÊNCIA] Não foi possível baixar {nome_arquivo}. Tente em outro horário.")
                
                sleep(0.5)

    print("\nProcesso concluído!")

extrair_ifdata_robusto('./dados_bcb_olinda')

Iniciando Extração de Alta Performance: /home/inovar-para-pessoas-negras/Área de trabalho/gustavo/bank-distress-foretoken/dados_bcb_olinda
------------------------------------------------------------
[-] Vazio: 200503 - Resumo
[-] Vazio: 200503 - Ativo
[-] Vazio: 200503 - Passivo
[-] Vazio: 200503 - Resultado


KeyboardInterrupt: 

In [2]:
import pandas as pd
import glob
import os

def processar_e_unir_ifdata(pasta_dados):
    print("Iniciando a leitura de todos os arquivos...")
    
    arquivos = glob.glob(os.path.join(pasta_dados, "IFDATA_*.csv"))
    
    if not arquivos:
        print("Nenhum arquivo encontrado na pasta.")
        return None

    lista_dfs = []

    # 1. Lê todos os arquivos e coloca numa lista
    for arquivo in arquivos:
        print(f"Lendo: {os.path.basename(arquivo)}")
        df = pd.read_csv(arquivo, sep=';', dtype={'CodInst': str, 'AnoMes': str})
        lista_dfs.append(df)

    print("\nEmpilhando os dados (Concatenando)...")
    # 2. Junta todos os arquivos um embaixo do outro
    df_completo = pd.concat(lista_dfs, ignore_index=True)

    print("Limpando e estruturando os nomes das contas contábeis...")
    # 3. Limpeza e prefixos
    if 'NomeColuna' in df_completo.columns:
        df_completo['NomeColuna'] = df_completo['NomeColuna'].str.replace('\n', ' ', regex=False)
        df_completo['NomeColuna'] = df_completo['NomeColuna'].str.replace(r'\s+', ' ', regex=True).str.strip()
        
        if 'NomeRelatorio' in df_completo.columns:
            df_completo['NomeColuna'] = df_completo['NomeRelatorio'] + " - " + df_completo['NomeColuna']

    print("Girando a tabela para o formato de Inteligência Artificial (Pivot)...")
    # 4. O PIVOT ÚNICO E PODEROSO: Transforma as contas em colunas de uma só vez
    # Como todos os dados já estão empilhados, não haverá colisão.
    df_mestre = df_completo.pivot_table(
        index=['CodInst', 'AnoMes'], 
        columns='NomeColuna', 
        values='Saldo', 
        aggfunc='first'
    ).reset_index()

    # Preenche o que o banco não reportou com 0
    df_mestre = df_mestre.fillna(0)

    # 5. Salva o Dataset
    caminho_saida = os.path.join(pasta_dados, "DATASET_MESTRE_IA.csv")
    df_mestre.to_csv(caminho_saida, index=False, sep=';', encoding='utf-8-sig')
    
    print("-" * 60)
    print(f"SUCESSO! Tabela final criada com {df_mestre.shape[0]} registros e {df_mestre.shape[1]} colunas contábeis.")
    print(f"Arquivo salvo em: {caminho_saida}")
    
    return df_mestre

# ==========================================
# Executar
# ==========================================
dataset_final = processar_e_unir_ifdata('./dados_bcb_olinda')

Iniciando a leitura de todos os arquivos...
Lendo: IFDATA_202103_Passivo.csv
Lendo: IFDATA_202212_Resultado.csv
Lendo: IFDATA_201903_Ativo.csv
Lendo: IFDATA_202012_Resumo.csv
Lendo: IFDATA_202403_Resumo.csv
Lendo: IFDATA_201506_Ativo.csv
Lendo: IFDATA_202403_Resultado.csv
Lendo: IFDATA_202312_Resultado.csv
Lendo: IFDATA_202406_Passivo.csv
Lendo: IFDATA_202309_Resumo.csv
Lendo: IFDATA_201909_Ativo.csv
Lendo: IFDATA_201603_Passivo.csv
Lendo: IFDATA_201406_Resumo.csv
Lendo: IFDATA_201809_Resultado.csv
Lendo: IFDATA_202006_Resultado.csv
Lendo: IFDATA_201809_Resumo.csv
Lendo: IFDATA_202206_Resultado.csv
Lendo: IFDATA_202409_Ativo.csv
Lendo: IFDATA_201709_Ativo.csv
Lendo: IFDATA_201503_Ativo.csv
Lendo: IFDATA_201909_Resumo.csv
Lendo: IFDATA_201612_Ativo.csv
Lendo: IFDATA_201709_Passivo.csv
Lendo: IFDATA_202412_Ativo.csv
Lendo: IFDATA_201512_Resultado.csv
Lendo: IFDATA_202106_Ativo.csv
Lendo: IFDATA_202103_Ativo.csv
Lendo: IFDATA_202409_Passivo.csv
Lendo: IFDATA_201412_Passivo.csv
Lendo: IFDA

In [3]:
import pandas as pd
import os

def aplicar_gabarito_falencias(caminho_dados_ifdata, caminho_dados_regesp):
    print("1. Carregando a lista oficial de falências (Regesp)...")
    
    # Lê o arquivo de regimes especiais forçando o CNPJ a ser texto (para não perder zeros à esquerda)
    df_regesp = pd.read_csv(caminho_dados_regesp, sep=';', dtype={'Cnpj': str}, encoding='utf-8', on_bad_lines='skip')
    
    # Preenche zeros à esquerda caso o Excel/CSV tenha comido, garantindo 14 dígitos
    df_regesp['Cnpj'] = df_regesp['Cnpj'].str.zfill(14)
    
    # A MÁGICA 1: O CodInst do IFDATA são os 8 primeiros dígitos do CNPJ base
    df_regesp['CodInst'] = df_regesp['Cnpj'].str[:8]
    
    # A MÁGICA 2: Converter a data "YYYY-MM-DD" para o formato "YYYYMM" numérico do IFDATA
    df_regesp['DataInicioRegime'] = pd.to_datetime(df_regesp['DataInicioRegime'], errors='coerce')
    # Removemos linhas sem data
    df_regesp = df_regesp.dropna(subset=['DataInicioRegime']) 
    df_regesp['AnoMes_Quebra'] = df_regesp['DataInicioRegime'].dt.strftime('%Y%m').astype(int)

    # Vamos filtrar apenas de 2014 em diante, já que seus dados IFDATA começam aí
    df_regesp = df_regesp[df_regesp['AnoMes_Quebra'] >= 201400]
    
    print(f" -> Encontrados {len(df_regesp)} registros de regimes especiais de 2014 para cá.")

    print("\n2. Carregando o Dataset Mestre Histórico (IFDATA)...")
    df_mestre = pd.read_csv(caminho_dados_ifdata, sep=';', dtype={'CodInst': str, 'AnoMes': int})
    
    # Cria a coluna Target, assumindo inicialmente que todos são bancos saudáveis (0)
    df_mestre['Falencia_Target'] = 0
    
    print("\n3. Cruzando os dados e aplicando os rótulos de risco...")
    bancos_afetados = 0
    
    for _, linha in df_regesp.iterrows():
        codigo = linha['CodInst']
        data_quebra = linha['AnoMes_Quebra']
        nome_banco = linha['NomeIf']
        
        # Se o banco que faliu existe no nosso dataset mestre
        if codigo in df_mestre['CodInst'].values:
            bancos_afetados += 1
            
            # Marca '1' em todos os trimestres ANTERIORES ou IGUAIS à data da quebra
            mascara_risco = (df_mestre['CodInst'] == codigo) & (df_mestre['AnoMes'] <= data_quebra)
            df_mestre.loc[mascara_risco, 'Falencia_Target'] = 1
            
            print(f" [+] Marcado: {nome_banco[:30]}... (Quebrou em {data_quebra})")

    # 4. Salvar o resultado
    caminho_saida = caminho_dados_ifdata.replace('.csv', '_ROTULADO.csv')
    df_mestre.to_csv(caminho_saida, index=False, sep=';', encoding='utf-8-sig')
    
    print("-" * 60)
    print("RESUMO DA ROTULAÇÃO:")
    print(f"Total de Instituições falidas encontradas no IFDATA: {bancos_afetados}")
    print(f"Linhas Saudáveis (0): {len(df_mestre[df_mestre['Falencia_Target'] == 0])}")
    print(f"Linhas com Doença/Falência (1): {len(df_mestre[df_mestre['Falencia_Target'] == 1])}")
    print(f"\nArquivo final pronto para a IA salvo em: {caminho_saida}")

# ==========================================
# Executar
# Substitua pelos caminhos reais dos seus arquivos
# ==========================================
caminho_ifdata = './dados_bcb_olinda/DATASET_MESTRE_IA.csv'
caminho_regesp = './Regesp_Internet_Exportar-1.csv' # Aponte para onde você salvou este arquivo

aplicar_gabarito_falencias(caminho_ifdata, caminho_regesp)

1. Carregando a lista oficial de falências (Regesp)...
 -> Encontrados 59 registros de regimes especiais de 2014 para cá.

2. Carregando o Dataset Mestre Histórico (IFDATA)...

3. Cruzando os dados e aplicando os rótulos de risco...
 [+] Marcado: COOPERATIVA DE CREDITO RURAL H... (Quebrou em 201703)
 [+] Marcado: COOPERATIVA DE CRÉDITO MÚTUO D... (Quebrou em 201801)
 [+] Marcado: COOPERATIVA DE CRÉDITO RURAL C... (Quebrou em 201809)
 [+] Marcado: COOPERATIVA DE ECONOMIA E CRÉD... (Quebrou em 202302)
 [+] Marcado: WILL FINANCEIRA S.A. CRÉDITO, ... (Quebrou em 202601)
------------------------------------------------------------
RESUMO DA ROTULAÇÃO:
Total de Instituições falidas encontradas no IFDATA: 5
Linhas Saudáveis (0): 61019
Linhas com Doença/Falência (1): 84

Arquivo final pronto para a IA salvo em: ./dados_bcb_olinda/DATASET_MESTRE_IA_ROTULADO.csv


In [4]:
import pandas as pd
import numpy as np
import glob
import os
from sklearn.preprocessing import StandardScaler

def criar_matriz_gnn_robusta(pasta_dados):
    print("1. Iniciando o Pipeline de Processamento para GNN...")
    todos_ficheiros = glob.glob(os.path.join(pasta_dados, "IFDATA_*.csv"))
    
    if not todos_ficheiros:
        print("Erro: Nenhum ficheiro encontrado.")
        return
    
    lista_dfs = []

    print("2. Lendo, higienizando e empilhando os dados temporais...")
    for ficheiro in todos_ficheiros:
        try:
            nome_base = os.path.basename(ficheiro).replace(".csv", "")
            partes = nome_base.split("_")
            if len(partes) < 3: continue
                
            relatorio = partes[2] 
            
            df = pd.read_csv(ficheiro, sep=';', dtype=str)
            
            if 'Item' in df.columns and 'valor' in df.columns:
                df['Item_Completo'] = relatorio + "_" + df['Item'].str.strip()

                df['valor'] = df['valor'].astype(str).str.replace('.', '', regex=False).str.replace(',', '.', regex=False)
                df['valor'] = pd.to_numeric(df['valor'], errors='coerce') # O que for lixo vira NaN
                
                lista_dfs.append(df[['Data', 'CnpjInst', 'NomeInst', 'Item_Completo', 'valor']])
        except Exception as e:
            print(f" [!] Erro ao processar {ficheiro}: {e}")

    df_empilhado = pd.concat(lista_dfs, ignore_index=True)
    
    print("3. Removendo duplicatas e anomalias do BCB...")
    df_empilhado = df_empilhado.drop_duplicates(subset=['Data', 'CnpjInst', 'Item_Completo'], keep='last')

    print("4. Pivotando para a Visão de Contexto Geral (Nós do Grafo)...")
    df_mestre = df_empilhado.pivot(
        index=['Data', 'CnpjInst', 'NomeInst'], 
        columns='Item_Completo', 
        values='valor'
    ).reset_index()

    print("5. Aplicando Filtro de Esparsidade (Remoção de Ruído)...")
    limite_nulos = len(df_mestre) * 0.60 
    df_mestre = df_mestre.dropna(thresh=limite_nulos, axis=1)

    df_mestre = df_mestre.fillna(0)
    
    print("6. Normalizando as Escalas Matemáticas (StandardScaler)...")
    cols_identificadoras = ['Data', 'CnpjInst', 'NomeInst']
    cols_numericas = [col for col in df_mestre.columns if col not in cols_identificadoras]
    
    scaler = StandardScaler()
    df_mestre[cols_numericas] = scaler.fit_transform(df_mestre[cols_numericas])

    caminho_saida = os.path.join(pasta_dados, "NODE_FEATURES_GNN_2014_2024.csv")
    df_mestre.to_csv(caminho_saida, index=False, sep=';', encoding='utf-8-sig')
    
    print("-" * 60)
    print(f"SUCESSO! Matriz Mestre Robusta criada.")
    print(f" - Instâncias (Nós no tempo): {df_mestre.shape[0]}")
    print(f" - Features Úteis (após corte de esparsidade): {len(cols_numericas)}")
    print(f"Arquivo pronto para a GNN salvo em: {caminho_saida}")

criar_matriz_gnn_robusta('./dados_bcb_olinda')

1. Iniciando o Pipeline de Processamento para GNN...
2. Lendo, higienizando e empilhando os dados temporais...


ValueError: No objects to concatenate

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt

def treinar_modelo_falencia(caminho_dataset_rotulado):
    print("1. Carregando os dados para a IA...")
    df = pd.read_csv(caminho_dataset_rotulado, sep=';')
    
    df = df.fillna(0)
    
    print(f"Total de registros: {len(df)}")
    
    ano_corte = 202000 
    
    df_treino = df[df['AnoMes'] < ano_corte]
    df_teste = df[df['AnoMes'] >= ano_corte]
    
    print(f"\n2. Dividindo os dados no tempo:")
    print(f" -> Treino (Passado - Até 2019): {len(df_treino)} balanços")
    print(f" -> Teste  (Futuro - 2020 a 2024): {len(df_teste)} balanços")

    colunas_ignoradas = ['CodInst', 'AnoMes', 'Falencia_Target']
    colunas_features = [col for col in df.columns if col not  in colunas_ignoradas]

    X_treino = df_treino[colunas_features]
    y_treino = df_treino['Falencia_Target']
    
    X_teste = df_teste[colunas_features]
    y_teste = df_teste['Falencia_Target']
    
    print("\n3. Treinando a Inteligência Artificial (Random Forest)...")

    modelo = RandomForestClassifier(
        n_estimators=100, 
        random_state=42, 
        class_weight='balanced',
        max_depth=10, 
        n_jobs=-1     
    )
    
    modelo.fit(X_treino, y_treino)
    
    print("\n4. Aplicando a Prova Final (Testando no Futuro)...")
    previsoes = modelo.predict(X_teste)
    probabilidades = modelo.predict_proba(X_teste)[:, 1] 

    print("\n" + "="*50)
    print("BOLETIM DE NOTAS DA IA (DADOS INÉDITOS)")
    print("="*50)
    
    auc = roc_auc_score(y_teste, probabilidades)
    print(f"Métrica ROC-AUC: {auc:.4f}\n")
    
    print("Matriz de Confusão:")
    matriz = confusion_matrix(y_teste, previsoes)
    print(matriz)
    
    print("\nRelatório de Classificação (Precision e Recall):")
    print(classification_report(y_teste, previsoes))
    
    print("="*50)
    importancias = pd.DataFrame({
        'Conta_Contabil': colunas_features,
        'Importancia_IA': modelo.feature_importances_
    }).sort_values(by='Importancia_IA', ascending=False)
    
    print("\nTop 5 Contas Contábeis que mais sinalizam Risco de Falência:")
    print(importancias.head(5))

    return modelo, colunas_features

modelo_treinado, features = treinar_modelo_falencia('./dados_bcb_olinda/DATASET_MESTRE_IA_ROTULADO.csv')

1. Carregando os dados para a IA...


/tmp/ipykernel_8048/297286047.py:8: DtypeWarning: Columns (0: CodInst) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(caminho_dataset_rotulado, sep=';')


Total de registros: 61099

2. Dividindo os dados no tempo:
 -> Treino (Passado - Até 2019): 34610 balanços
 -> Teste  (Futuro - 2020 a 2024): 26489 balanços

3. Treinando a Inteligência Artificial (Random Forest)...

4. Aplicando a Prova Final (Testando no Futuro)...

BOLETIM DE NOTAS DA IA (DADOS INÉDITOS)
Métrica ROC-AUC: 0.8319

Matriz de Confusão:
[[26446    27]
 [   13     3]]

Relatório de Classificação (Precision e Recall):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     26473
           1       0.10      0.19      0.13        16

    accuracy                           1.00     26489
   macro avg       0.55      0.59      0.56     26489
weighted avg       1.00      1.00      1.00     26489


Top 5 Contas Contábeis que mais sinalizam Risco de Falência:
                                       Conta_Contabil  Importancia_IA
16  Ativo - TVM e Instrumentos Financeiros Derivat...        0.119241
39                   Passivo - Patri